# C-test ONE · Calibrator

*Ctest-Calibrator-v1.4.1*

Measures the constants of each reference machine on a fixed set of probes: bits per
character ($\gamma$), the cost of the shortest exception clause ($\delta$), and the fixed and
per-term cost of listing a sequence ($c_0$ and $\beta$). It also writes the pooled constants
that the generator reads.

| reads | writes |
|---|---|
| (nothing) | `ctest_calibration.json` |

**Needs** a GPU (16 GB is enough for 4-bit models up to 14B) and a Hugging Face token for gated
models (Llama, Gemma). About half an hour to an hour per machine.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


## 1 · Install

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

import os

# Only on Colab. Kaggle ships packages that make `import google.colab` succeed,
# so the import is not evidence of anything; the filesystem is.
if os.path.exists("/kaggle"):
    print("on Kaggle: no Drive to mount, and none needed")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as _e:
        print("not on Colab, so no Drive to mount:", _e)


## 2 · Preamble

In [ ]:
import os, sys, re, json, time, glob, math, shutil, random, hashlib, platform
import collections, statistics as st, datetime, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

BUILD = "Ctest-Calibrator-v1.4.1"
print("=" * 72)
print(f"{BUILD}")
print("=" * 72)

# The filesystem decides, not an import. Kaggle ships packages that make
# `import google.colab` succeed, which sent ENV to "colab" on Kaggle and killed
# the run on the unmounted-Drive guard.
if os.path.exists("/kaggle"):
    ENV = "kaggle"
elif os.path.isdir("/content"):
    ENV = "colab"
else:
    ENV = "local"
# Two separate places, because they are separate things and always were.
DRIVE_FOLDER   = "ctest"        # Colab: /content/drive/MyDrive/ctest
KAGGLE_DATASET = "ctest-one-data"    # Kaggle: the dataset to read from and push to.
                                # A Kaggle dataset is a FOLDER of files, not a
                                # single file.

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
# OUTDIR is a second place to copy finished files to. On Colab that is Drive.
# On Kaggle there is no such place: an attached dataset is READ-ONLY at
# /kaggle/input, and the working directory is the only writable one and is
# already where files are written. Copying a file onto itself raises
# SameFileError, so OUTDIR stays "." and persistence is the dataset push in the
# cell below.
if ENV == "colab" and os.path.isdir("/content/drive/MyDrive"):
    OUTDIR = DRIVE
else:
    OUTDIR = "."
print(f"environment      {ENV}   python {platform.python_version()}")

# shutil.copy writes a FILE when its destination is not an existing directory, so
# without this every save lands in a file named after the folder and the results
# leave with the session.
if OUTDIR != ".":
    if os.path.exists(OUTDIR) and not os.path.isdir(OUTDIR):
        raise SystemExit(f"{OUTDIR} exists but is a file. Rename or delete it.")
    os.makedirs(OUTDIR, exist_ok=True)
elif ENV == "colab":
    raise SystemExit("Drive is not mounted, so anything written would be lost. Run\n"
                     "    from google.colab import drive; drive.mount('/content/drive')")
if ENV == "kaggle":
    print(f"outputs go to    the working directory, then pushed to the "
          f"{KAGGLE_DATASET} dataset")
    print( "                 (a dataset cannot be written to in place; see the "
           "persistence cell)")
if ENV != "kaggle":
    print(f"outputs go to    {OUTDIR}")

HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        _p = torch.cuda.get_device_properties(0)
        print(f"GPU              {_p.name}, {_p.total_memory/2**30:.0f} GiB")
    else:
        print("GPU              none found. This notebook prices programs under a")
        print("                 reference machine, so it will be very slow on CPU.")
except Exception:
    print("GPU              torch not installed")

def _secret(n):
    """Look for a secret, and SAY what each provider reported.

    An earlier version swallowed the Kaggle error and fell through to the Colab
    branch, which raises on Kaggle, so the result was a bare None and nothing
    said why. Kaggle secrets are PER NOTEBOOK: one attached to another session
    does not carry over."""
    if os.environ.get(n):
        return os.environ[n]
    notes = []
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(n)
        if v:
            return v
        notes.append("kaggle: returned nothing")
    except ImportError:
        pass                                  # not on Kaggle, which is fine
    except Exception as e:
        notes.append(f"kaggle: {type(e).__name__}: {str(e)[:90]}")
    try:
        from google.colab import userdata
        v = userdata.get(n)
        if v:
            return v
        notes.append("colab: returned nothing")
    except ImportError:
        pass
    except Exception as e:
        notes.append(f"colab: {type(e).__name__}: {str(e)[:90]}")
    for m in notes:
        print(f"                 {m}")
    return None

_key = _secret("GOOGLE_API_KEY")
if _key: os.environ["GOOGLE_API_KEY"] = _key
print(f"GOOGLE_API_KEY   {'found' if _key else 'not needed by this notebook'}")

# The dataset, from a secret when there is one, so two accounts can work in
# parallel on separate copies with this notebook byte-identical in both.
_ds = _secret("KAGGLE_DATASET")
if _ds:
    KAGGLE_DATASET = _ds.strip()
print(f"KAGGLE_DATASET   {KAGGLE_DATASET}"
      + ("   from the secret" if _ds else "   the built-in default"))

# The Hub token has to be put INTO the environment before transformers is asked
# for a model. A Kaggle secret is not an environment variable: it lives behind
# UserSecretsClient and is invisible to huggingface_hub until something fetches
# it, which is why a notebook with HF_TOKEN in its secrets still reports
# "unauthenticated requests to the HF Hub". Unauthenticated still works for a
# public model; it is rate limited and slower, and it fails outright on a gated
# one.
#
# Both names are set. huggingface_hub reads HF_TOKEN, and some versions of
# transformers and datasets still look for HUGGING_FACE_HUB_TOKEN.
_hf = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN") \
      or _secret("HUGGINGFACE_TOKEN")
if _hf:
    os.environ["HF_TOKEN"] = _hf
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = _hf
    print(f"HF_TOKEN         found, {len(_hf)} chars, set for the Hub")
else:
    print( "HF_TOKEN         not found. Downloads are unauthenticated: rate")
    print( "                 limited, and gated models will not resolve. On")
    print( "                 Kaggle add it under Add-ons -> Secrets with the")
    print( "                 toggle ON for this notebook; on Colab use the key")
    print( "                 panel. The secret must be named HF_TOKEN.")

_PRINT_LOCK = threading.Lock()

def log(*a, **kw):
    with _PRINT_LOCK:
        print(*a, flush=True, **kw)

def stage(t):
    with _PRINT_LOCK:
        print(f"\n{'='*72}\n== {t}\n{'='*72}", flush=True)

def find(f):
    for p in [f, f"/content/{f}", f"{DRIVE}/{f}"] + glob.glob(f"/kaggle/input/**/{f}", recursive=True):
        if os.path.exists(p): return p
    return None

def load(name, default):
    p = find(name)
    if not p:
        log(f"  {name}: absent, starting empty"); return default
    log(f"  {name}: loaded from {p}")
    return json.load(open(p))


## 3 · The run

In [ ]:

# ALL FOUR, on probe set v2. Three of them already have a block from v1, and
# those numbers are not comparable with these: v1's sequences were all cycles
# or counts, so beta came out an order of magnitude low. CALIB_REMEASURE below
# is therefore ON, and each block is replaced rather than added to. Nothing
# from v1 survives.
# One machine: the second reference machine for the battery. Qwen3-14B rather
# than a 7B from another family, because it has to be UNAMBIGUOUSLY stronger
# than Qwen3-8B for the experiment to mean anything, and in 4-bit it is about
# 9 GB so it still fits the T4 that the long annotation runs need.
#
# A different family, OLMo, is worth doing later for the same battery: it would
# say whether the constants move with capability or with the house.
# A THIRD REFERENCE MACHINE, FROM ANOTHER FAMILY. Qwen3-8B and Qwen3-14B are
# the same lineage and the same tokeniser, so every difference between them
# confounds size with nothing else. A machine trained elsewhere separates
# "weaker" from "trained differently", which is what decides whether Kt is a
# scale or a house style.
#
# CHECK THE REPO NAME BEFORE RUNNING. Microsoft published Phi-4-reasoning and
# Phi-4-reasoning-plus in 2025 and the exact string may differ from the one
# below; plain `microsoft/phi-4` is the NON-reasoning model and is not what is
# wanted here. Open the model page, copy the identifier, and paste it.
#
# LOADED IN NF4 LIKE THE OTHERS, not AWQ. c_M(p) in bits depends on the
# quantisation, so a machine in a different format would not be on the same
# footing and only its ORDERING could be compared, not its values.
MACHINES = [                     # the six machines of Table 29
    "Qwen/Qwen3-14B",
    "Qwen/Qwen3-8B",
    "microsoft/Phi-4-reasoning",
    "google/gemma-3-4b-it",
    "allenai/OLMo-2-1124-7B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
]

# WHICH MACHINES THE POOLED BLOCK IS BUILT FROM. The generator reads `pooled`
# to size sequences and place rival prefixes, so this set decides how every
# item after it is built. Naming them means a machine calibrated later cannot
# change it by accident.
#
# All six are in, Phi-4-reasoning included, because adding it moved NOTHING:
# it is not the extreme on any pooled field. beta_gate is the largest
# beta_high and that is OLMo's 23.53 against Phi-4's 16.30; delta_gen is the
# largest clause cost and that is Qwen3-8B's 22.7 against 16.3; bits_per_char
# is gemma's 3.31; beta_listing and c0_listing are Qwen3-14B's 4.11 and
# Llama's 116.6. Every pooled value is identical over five machines and over
# six, so the v17 battery was generated under these numbers whichever way the
# membership is counted.
#
# A seventh machine is a different matter and should be checked before it is
# added: if it IS the extreme on some field, the battery before and after it
# would be sized under different rules, and `pooled_before` would be the only
# record.
#
# None would pool over every machine on the current probe set, which is how it
# behaved before and is what this replaces.
POOL_MACHINES = [                # the pool the battery was generated with (Table 29)
    "Qwen/Qwen3-8B",
    "google/gemma-3-4b-it",
    "allenai/OLMo-2-1124-7B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
]
# already in the file:
#   Qwen/Qwen3-8B                     bits/char 1.96, beta 3.4, c0 135.2
# listed but never completed:
#   google/gemma-3-4b-it, allenai/OLMo-2-1124-7B-Instruct,
#   meta-llama/Llama-3.1-8B-Instruct
# the three already in the file, for reference:
#   Qwen/Qwen3-8B
#   google/gemma-3-4b-it
#   allenai/OLMo-2-1124-7B-Instruct

LOAD_4BIT   = True
DISK_MIN_GB = 12      # a T4 session has about 21 GB and an 8B model in 4-bit
                      # takes most of it, so they are done ONE AT A TIME and
                      # the weights are deleted between. Below this the next
                      # download would fail halfway.
DROP_CACHE  = True
SINGLE_MACHINE_OK = True  # silences the "only one machine" warning, which is
                      # right in general and wrong for a top-up run

BETA_MIN_R2 = 0.80
PROBE_SET_ID = "fixed-v2"      # bump this if the probes below ever change,
                               # or old and new numbers will be compared as
                               # though they were the same measurement
CALIB_REMEASURE = True         # True replaces a reference block already there.
                               # ON, because three machines carry probe set v1
                               # numbers and mixing the two sets in one pooled
                               # value would be meaningless. Turn it off once
                               # every machine is on the current set.
CALIBRATION = "ctest_calibration.json"

# The listing, exactly as the unquestionability pass writes it. If that ever
# changes, c0 measured here stops describing the same string.
# Only cost_many is used here, which prices unconditionally. PRICE_TPL is
# defined because the Machine class carries cost_given for the other
# notebooks, and an unbound name in a class is a trap for whoever reuses it.
PRICE_TPL = ("Here is the beginning of a sequence:\n{terms}\n"
             "A rule that generates it:")

PRINT_TPL    = "Print: "
LISTING_TAIL = ", then repeat that list forever"

BATCH_ROWS, BATCH_TOKENS = 8, 3000

print(f"machines      {len(MACHINES)}, one at a time")
for m in MACHINES: print(f"              {m}")
print(f"probes        {PROBE_SET_ID}, fixed and shipped with this notebook")
print(f"writes        {CALIBRATION}, under machines.<name>.reference")


## 3b · Can we fetch these?

A gated repo fails at download time, several minutes in and after a machine has
already been pulled.
One HTTP call each settles it first.

Llama and Gemma are gated.
Access is granted per **family** and per **Hugging
Face account**, so accepting the Llama 3.1 licence covers every 3.1 repo but
not 3.2, and it is the account behind `HF_TOKEN` that matters rather than the
token.
The other trap is that a *fine-grained* token needs "Read access to the
contents of all public gated repos you can access" ticked, or it returns a 403
that looks exactly like not having been granted access.


In [ ]:
# ---------------------------------------------------------------------------
# CAN WE ACTUALLY FETCH THESE?
#
# A gated repo fails at download time, which on this notebook is several
# minutes in and after a machine has already been pulled. Checking first costs
# one HTTP call each and no bytes.
#
# Llama and Gemma are gated. Access is granted per FAMILY and per Hugging Face
# ACCOUNT, so accepting the Llama 3.1 licence covers every 3.1 repo but not
# 3.2, and it is the account behind HF_TOKEN that matters, not the token.
#
# The other trap: a FINE-GRAINED token needs "Read access to the contents of
# all public gated repos you can access" ticked, or it returns a 403 that looks
# exactly like not having been granted access. A classic read token has it.

STOP_IF_GATED = True     # False checks, reports, and carries on with the rest

stage("CHECKING ACCESS")
from huggingface_hub import HfApi, model_info, hf_hub_download

_api = HfApi()

def _can_fetch(repo):
    if hasattr(_api, "auth_check"):
        _api.auth_check(repo)          # raises when it cannot be fetched
        return
    hf_hub_download(repo, "config.json")

_ok, _bad = [], []
for _m in MACHINES:
    try:
        _can_fetch(_m)
        _g = getattr(model_info(_m), "gated", None)
        log(f"  ok       {_m}" + (f"   (gated: {_g}, and you are through)"
                                  if _g else ""))
        _ok.append(_m)
    except Exception as _e:
        _msg = str(_e)
        _kind = ("gated" if ("403" in _msg or "gated" in _msg.lower() or
                             "awaiting" in _msg.lower() or
                             "GatedRepo" in type(_e).__name__)
                 else "missing" if ("404" in _msg or
                                   "RepositoryNotFound" in type(_e).__name__)
                else "other")
        _why = {"gated": "gated, and this account has not been granted access",
                "missing": "not found. Check the exact name and its "
                           "capitalisation: Llama-3.1-8B-Instruct is a "
                           "different repo from Meta-Llama-3.1-8B-Instruct",
                "other": _msg[:90]}[_kind]
        log(f"  BLOCKED  {_m}")
        log(f"           {_why}")
        _bad.append((_m, _kind))

if _bad:
    log(f"\n  {len(_bad)} of {len(MACHINES)} cannot be fetched.")
    _g = [m for m, k in _bad if k == "gated"]
    _n = [m for m, k in _bad if k == "missing"]
    if _g:
        log(f"\n  GATED, so accept the licence:")
        log(f"    Sign in to huggingface.co with the account behind HF_TOKEN, "
            f"then open each of these and fill in the box at the top.")
        for _m in _g:
            log(f"      https://huggingface.co/{_m}")
        log(f"    Meta and Google usually approve within minutes. Access is "
            f"granted per FAMILY, so Llama 3.1 covers every 3.1 repo.")
        log(f"    If it still fails afterwards the token is the problem: a "
            f"fine-grained token needs 'Read access to the contents of all")
        log(f"    public gated repos you can access'. A classic read token "
            f"has it already.")
    if _n:
        log(f"\n  NOT FOUND, so the name is wrong. Nothing to accept:")
        for _m in _n:
            log(f"      {_m}")
    log(f"\n  Ungated alternatives from other families, if you would rather "
        f"not wait: mistralai/Ministral-8B-Instruct-2410,")
    log(f"  microsoft/Phi-4-mini-instruct, allenai/OLMo-2-1124-7B-Instruct. "
        f"The point is family spread, not any particular model.")
    if STOP_IF_GATED:
        raise SystemExit(
            f"stopping: {len(_bad)} machine(s) cannot be fetched. Fix the "
            f"access, drop them from MACHINES, or set STOP_IF_GATED = False "
            f"to run with the {len(_ok)} that work.")
    MACHINES = _ok
    log(f"\n  STOP_IF_GATED is off, carrying on with {len(MACHINES)}.")

if len(MACHINES) < 2 and not globals().get("SINGLE_MACHINE_OK"):
    log(f"\n  Only {len(MACHINES)} machine(s). The whole point of this "
        f"notebook is the SPREAD across machines: with one there is no way to")
    log(f"  tell whether a constant belongs to reference machines as a class "
        f"or to that machine alone.")
log(f"\n  {len(MACHINES)} machine(s) ready.")


## 3c · Persistent state

In [ ]:
import subprocess

STATE_PATTERNS = ("ctest_*.json", "ctest_*.csv")

STATE_DATASET = ""        # "owner/slug"; empty derives it from KAGGLE_DATASET
STATE_PUSH_EVERY = 0      # 0 pushes only at the end of a run

def _kaggle_ready():
    if ENV != "kaggle":
        return False
    missing = []
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        v = _secret(k)
        if v:
            os.environ[k] = v
        else:
            missing.append(k)
    if missing:
        log(f"  state persistence OFF, missing secret(s): {', '.join(missing)}")
        log("     Add-ons -> Secrets -> add them from your kaggle.json, and turn")
        log("     the toggle on for this notebook. Results still appear in the")
        log("     version output; only state across versions is lost.")
        return False
    return True

KAGGLE_OK = _kaggle_ready()

def _slug():
    if STATE_DATASET:
        return STATE_DATASET
    return f"{os.environ.get('KAGGLE_USERNAME','unknown')}/{KAGGLE_DATASET.lower()}"

def state_files():
    """Every file the push should carry, newest copy of each name.

    Local first, since this run may have just written it. Then whatever is in
    the attached dataset and not local, so another notebook's output survives a
    push from this one."""
    chosen, source = {}, {}
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(pat)):
            chosen[os.path.basename(f)] = f
            source[os.path.basename(f)] = "this run"
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)):
            b = os.path.basename(f)
            if b not in chosen:
                chosen[b] = f
                source[b] = "carried forward"
    return chosen, source

FRESH_DIR = "/kaggle/working/_fresh"

def _dataset_names():
    """Filenames actually IN the dataset, from the refreshed download.

    NOT from /kaggle/input. The mount is fixed when the session starts, so it
    can be older than the dataset, and comparing the mount against a push
    assembled from the mount compares a thing with itself and can never fire.
    That was the whole failure: the dataset held ctest_metrics.json, the mount
    did not, and a push assembled from the mount deleted it.

    If the refresh did not run or failed, this returns nothing and the guard
    stays quiet, because there is no trustworthy list to compare against.
    """
    out = set()
    if not os.path.isdir(FRESH_DIR):
        return out
    for pat in STATE_PATTERNS:
        for f in glob.glob(os.path.join(FRESH_DIR, pat)):
            out.add(os.path.basename(f))
    return out

STATE_REFRESH = True

def state_refresh():
    """Download the current dataset version into the working directory.

    Cheap, a couple of megabytes, and it removes a whole class of confusion:
    after this the notebook is reading what is IN the dataset rather than what
    happened to be mounted.

    Never raises. A failure leaves /kaggle/input in place, which is the old
    behaviour, and says so.
    """
    if not (STATE_REFRESH and KAGGLE_OK):
        return
    slug = _slug()
    d = FRESH_DIR
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    try:
        r = subprocess.run(["kaggle", "datasets", "download", "-d", slug,
                            "-p", d, "--unzip"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            log(f"  could not refresh from {slug}: "
                f"{(r.stderr or r.stdout).strip()[:120]}")
            log(f"  falling back to whatever the session mounted, which may be "
                f"older than the dataset.")
            return
    except Exception as e:
        log(f"  could not refresh from {slug}: {str(e)[:100]}")
        return

    got, stale = [], []
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            b = os.path.basename(f)
            mounted = None
            for p in glob.glob(f"/kaggle/input/**/{b}", recursive=True):
                mounted = p; break
            if os.path.exists(b):
                continue
            shutil.copy(f, b)
            got.append(b)
            if mounted is None:
                stale.append(f"{b} (NOT mounted at all)")
            elif os.path.getsize(mounted) != os.path.getsize(f):
                stale.append(f"{b} (mounted copy is a different size)")
    log(f"  refreshed {len(got)} file(s) from {slug}: {', '.join(sorted(got))}")
    if stale:
        log(f"  the session's mount was OUT OF DATE for: {', '.join(stale)}")
        log(f"  That is why this matters: without the refresh those would have "
            f"been read stale or not at all.")

def state_save(msg="update"):
    """Push a new version of the state dataset. Never raises."""
    if not KAGGLE_OK:
        return
    chosen, source = state_files()
    if not chosen:
        log("  no state files to push yet"); return
    d = "/kaggle/working/_state"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    for b, f in chosen.items():
        shutil.copy(f, os.path.join(d, b))
    _in_dataset = _dataset_names()
    if not _in_dataset:
        log(f"  note: no refreshed copy of the dataset, so the push cannot be "
            f"checked for deletions. Set STATE_REFRESH = True.")
    _would_lose = _in_dataset - set(chosen)
    if _would_lose:
        log(f"  REFUSING TO PUSH. These are in the dataset and not in this "
            f"push, so the push would DELETE them:")
        for b in sorted(_would_lose):
            log(f"      {b}")
        log(f"  Most likely the session mounted an old version of the dataset. "
            f"STATE_REFRESH pulls the current one at startup; if that failed,")
        log(f"  the log above says why. Nothing has been pushed and nothing is "
            f"lost.")
        return

    carried = [b for b, s in source.items() if s == "carried forward"]
    log(f"  staging {len(chosen)} files"
        + (f", {len(carried)} carried forward from the dataset: "
           f"{', '.join(sorted(carried)[:4])}"
           f"{' ...' if len(carried) > 4 else ''}" if carried else ""))
    slug = _slug()
    json.dump({"title": slug.split("/")[-1], "id": slug,
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{d}/dataset-metadata.json", "w"))
    try:
        r = subprocess.run(["kaggle", "datasets", "version", "-p", d, "-m", msg,
                            "-r", "zip", "--dir-mode", "zip"],
                           capture_output=True, text=True, timeout=900)
        out = (r.stdout + r.stderr).strip().split("\n")[-1][:160]
        if "not found" in out.lower() or "404" in out:
            r = subprocess.run(["kaggle", "datasets", "create", "-p", d, "-r", "zip",
                                "--dir-mode", "zip"],
                               capture_output=True, text=True, timeout=900)
            out = (r.stdout + r.stderr).strip().split("\n")[-1][:160]
            log(f"  created dataset '{slug}'. Attach it as an input so the next"
                f" version can read it back.")
        log(f"  state push: {out}")
    except Exception as e:
        log(f"  state push failed ({str(e)[:80]}). The results are still in this"
            f" version's output.")

stage("PERSISTENT STATE")
if ENV != "kaggle":
    log(f"  not on Kaggle, so nothing to do. Outputs go to {OUTDIR}.")
elif KAGGLE_OK:
    subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)
    log(f"  credentials found for '{os.environ['KAGGLE_USERNAME']}'")
    log(f"  state dataset: {_slug()}")
    log(f"  pushes" + (f" every {STATE_PUSH_EVERY} items and" if STATE_PUSH_EVERY
        else "") + " at the end of the run")
_attached = sorted({os.path.basename(os.path.dirname(f))
                    for f in glob.glob("/kaggle/input/*/*")})
if ENV == "kaggle":
    if _attached:
        log(f"  datasets attached: {', '.join(_attached)}")
    else:
        log(f"  NO dataset attached. Add Input -> Datasets -> {KAGGLE_DATASET},")
        log(f"  or the loaders will not find the battery.")
_ch, _src = state_files()
if _ch:
    log(f"  state would carry {len(_ch)} files: {', '.join(sorted(_ch)[:6])}"
        + (" ..." if len(_ch) > 6 else ""))

state_refresh()


## 3d · What the run cost

In [ ]:
import csv, resource

RUNS_CSV = "ctest_runs.csv"

# Seed the local copy from wherever the file already lives, before anything is
# appended. Every other file is read back through find(), which globs
# /kaggle/input; this one was only ever appended, so on Kaggle it started empty
# each session and each run wrote a fresh one-row file. Worse, the dataset push
# prefers the local copy, so that one row REPLACED the accumulated history on
# every push. Rows were being lost, not merely not added.
_prev = find(RUNS_CSV)
if _prev and os.path.abspath(_prev) != os.path.abspath(RUNS_CSV):
    shutil.copy(_prev, RUNS_CSV)
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) carried in from {_prev}")
elif _prev:
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) already here")
else:
    print(f"run history       none yet, {RUNS_CSV} will be created")

_T_START = time.time()
_CPU_START = resource.getrusage(resource.RUSAGE_SELF).ru_utime + \
             resource.getrusage(resource.RUSAGE_SELF).ru_stime
_GPU_SECONDS = [0.0]      # wall clock with a model loaded; add to it when one is

def _gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return ""

def account(units=0, note=""):
    """Append one row for this run. Never raises: accounting must not stop work."""
    try:
        cpu = (resource.getrusage(resource.RUSAGE_SELF).ru_utime +
               resource.getrusage(resource.RUSAGE_SELF).ru_stime) - _CPU_START
        tk = dict(TOKENS) if "TOKENS" in globals() else {}
        row = dict(
            run_ended=datetime.datetime.now().isoformat(timespec="seconds"),
            build=BUILD, env=ENV, note=note,
            wall_s=round(time.time() - _T_START, 1),
            cpu_s=round(cpu, 1),
            gpu_s=round(_GPU_SECONDS[0], 1),
            gpu=_gpu_name(),
            units=units,
            model=(COMPOSER if "COMPOSER" in globals()
                   else REF_MACHINE if "REF_MACHINE" in globals() else ""),
            api_calls=tk.get("calls", 0),
            prompt_tokens=tk.get("in", 0),
            answer_tokens=tk.get("answer", 0),
            thinking_tokens=tk.get("think", 0),
            output_tokens=tk.get("out", 0),
            forward_passes=(REF.calls if "REF" in globals()
                            and hasattr(REF, "calls") else 0),
        )
        old_rows, old_cols = [], []
        if os.path.exists(RUNS_CSV):
            with open(RUNS_CSV, newline="") as f:
                rd = csv.DictReader(f)
                old_cols = list(rd.fieldnames or [])
                old_rows = list(rd)
        cols = old_cols + [c for c in row if c not in old_cols]
        if old_cols and cols != old_cols:
            log(f"  {RUNS_CSV} had {len(old_cols)} columns and this build writes "
                f"{len(row)}; rewriting with the union so no row is lost")
            with open(RUNS_CSV, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=cols, restval="")
                w.writeheader()
                for r in old_rows: w.writerow(r)
                w.writerow(row)
        else:
            new = not os.path.exists(RUNS_CSV)
            with open(RUNS_CSV, "a", newline="") as f:
                w = csv.DictWriter(f, fieldnames=cols or list(row), restval="")
                if new: w.writeheader()
                w.writerow(row)
        if OUTDIR != ".": shutil.copy(RUNS_CSV, OUTDIR)
        with open(RUNS_CSV) as f:
            _rows = max(0, sum(1 for _ in f) - 1)
        log(f"\n  cost of this run, appended to {RUNS_CSV} ({_rows} rows now):")
        log(f"    {row['wall_s']/60:.1f} min wall, {row['cpu_s']/60:.1f} min cpu"
            + (f", {row['gpu_s']/60:.1f} min gpu on {row['gpu']}"
               if row["gpu_s"] else ""))
        if row["api_calls"]:
            log(f"    {row['api_calls']} calls to {row['model']}: "
                f"{row['prompt_tokens']:,} prompt, {row['answer_tokens']:,} answer, "
                f"{row['thinking_tokens']:,} thinking")
        if row["forward_passes"]:
            log(f"    {row['forward_passes']} forward passes under {row['model']}")
        if units:
            log(f"    {units} units of work, so "
                f"{row['wall_s']/60/max(1,units):.1f} min each")
        log(f"    energy and CO2 are NOT recorded: they are derived later from "
            f"published figures, so they can be revised.")
    except Exception as e:
        log(f"  accounting failed ({str(e)[:70]}), which changes nothing else")


## 4 · Helpers

In [ ]:
import statistics as st, shutil, gc

def _q(v, p):
    v = sorted(v)
    return v[min(len(v) - 1, int(p * len(v)))] if v else None

def _fit(xs, ys):
    """Least squares, returning slope, intercept and R2."""
    n = len(xs); mx, my = sum(xs)/n, sum(ys)/n
    sxx = sum((x-mx)**2 for x in xs)
    if sxx == 0: return 0.0, my, 0.0
    b = sum((x-mx)*(y-my) for x, y in zip(xs, ys)) / sxx
    c = my - b*mx
    sst = sum((y-my)**2 for y in ys)
    ssr = sum((y - (b*x + c))**2 for x, y in zip(xs, ys))
    return b, c, (1 - ssr/sst if sst > 0 else 0.0)

def _disk_free_gb():
    return shutil.disk_usage("/").free / 1e9

def _unload():
    """Free the GPU and, if asked, the downloaded weights.

    One machine at a time is not a preference. The session has about 21 GB of
    disk and an 8B model in 4-bit needs most of it, so the second download
    fails unless the first is gone."""
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
    if DROP_CACHE:
        for d in ("/root/.cache/huggingface/hub",
                  os.path.expanduser("~/.cache/huggingface/hub")):
            if os.path.isdir(d):
                try:
                    shutil.rmtree(d); log(f"  cleared {d}")
                except Exception as e:
                    log(f"  could not clear {d}: {str(e)[:60]}")

def _load_calibration():
    p = find(CALIBRATION) or CALIBRATION
    try:
        d = json.load(open(p))
        return d
    except Exception:
        return {"_README": (
            "Constants describing each reference machine. `reference` is "
            "measured on the fixed probes in Ctest-Calibrator and belongs to "
            "the machine. `fitted` is measured by the unquestionability pass "
            "while pricing a battery and belongs to that battery, so it is "
            "kept per battery. `pooled` is what the generator reads: the "
            "conservative end across machines, an efficiency device and not "
            "part of any definition.")}

def _save_reference(blocks, pooled=None):
    """Merge reference blocks into the calibration file, in place.

    Called after EVERY machine, not once at the end. Four machines is forty
    minutes with a download apiece, and a session that dies on the third would
    otherwise take the first two with it.

    It only ever touches `reference` and `pooled`. `fitted`, which the
    unquestionability pass writes per battery, is left exactly as found.
    """
    ALL = _load_calibration()
    ALL.setdefault("machines", {})
    for name, r in blocks.items():
        ent = ALL["machines"].setdefault(name, {})
        old = ent.get("reference")
        if old and not CALIB_REMEASURE:
            log(f"    {name} already has a reference block from "
                f"{old.get('measured_on','?')}, left alone "
                f"(CALIB_REMEASURE = True to replace)")
            continue
        ent["reference"] = r
    if pooled:
        ALL["pooled"] = pooled
    with open(CALIBRATION, "w") as f:
        json.dump(ALL, f, indent=1)
    return ALL


## 5 · The reference machine

In [ ]:
class Machine:
    """The reference machine. Prices strings, and nothing else."""

    def __init__(self, name, four_bit=True):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch, self.name, self.calls = torch, name, 0
        log(f"  loading {name}" + ("  (4-bit)" if four_bit else "  (bfloat16)"))
        t0 = time.time()
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        kw = dict(trust_remote_code=True, device_map="auto")
        if four_bit:
            try:
                from transformers import BitsAndBytesConfig
                kw["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
            except Exception as e:
                log(f"  4-bit unavailable ({str(e)[:60]}), using bfloat16")
                kw["torch_dtype"] = torch.bfloat16
        else:
            kw["torch_dtype"] = torch.bfloat16
        self.model = AutoModelForCausalLM.from_pretrained(name, **kw)
        self.model.eval()
        self.dev = next(self.model.parameters()).device
        self._empty, self._memo = None, {}
        self._loaded_at = time.time()   # for the run accounting
        self.build = BUILD          # so a stale instance is not reused
        log(f"  loaded in {time.time()-t0:.0f}s on {self.dev}")

        places = collections.Counter(str(p.device) for p in self.model.parameters())
        log(f"  parameters by device: {dict(places)}")
        if any(d.startswith("cpu") or d == "meta" for d in places):
            log("  WARNING: part of the model is not on the GPU. Every forward")
            log("  pass will be slow. Use a smaller model, or LOAD_4BIT with a")
            log("  card that fits it.")

        # one timed pass, so the rate is known before 36 items are queued
        _t = time.time()
        _b = self.bits("the quick brown fox jumps over the lazy dog\n")
        log(f"  a short pass takes {time.time()-_t:.2f}s "
            f"({_b:.0f} bits for that probe)")

    def _ids(self, s):
        return self.tok(s, add_special_tokens=True).input_ids

    def _score(self, ids_batch, mask, starts):
        """Summed log2 P for each row, from `starts[r]` to that row's end.

        Two things matter for speed. Only the positions being scored are turned
        into probabilities, so the vocabulary-wide softmax is not computed for
        every token in the string. And the whole row is gathered in one
        operation with a single device-to-host transfer, instead of one
        `.item()` per token, which was a separate GPU sync each time."""
        torch = self.torch
        with torch.no_grad():
            logits = self.model(ids_batch, attention_mask=mask).logits
        out = []
        for r in range(ids_batch.shape[0]):
            n0, n1 = starts[r], int(mask[r].sum().item())
            if n1 <= n0:
                out.append(0.0); continue
            lg = logits[r, n0-1:n1-1, :].float()        # only what is scored
            tgt = ids_batch[r, n0:n1]
            lp = torch.log_softmax(lg, dim=-1)
            tot = lp.gather(1, tgt.unsqueeze(1)).sum().item()   # one transfer
            out.append(-tot / math.log(2))
        self.calls += ids_batch.shape[0]
        return out

    def bits_many(self, texts, condition=""):
        """-log2 P(text | condition) for several texts, batched.

        The texts are independent, so one forward pass covers a batch. Right
        padding is safe under causal attention: a token is never influenced by
        anything after it."""
        torch = self.torch
        if not texts:
            return []
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        ctx = self._ids(condition) if condition else self._ids("")
        rows, starts = [], []
        for t in texts:
            full = self._ids(condition + t)
            n = len(ctx)
            if full[:n] != ctx:                  # tokenisation did not line up
                n = 0
                while n < min(len(ctx), len(full)) and full[n] == ctx[n]:
                    n += 1
            rows.append(full); starts.append(max(n, 1))
        out, i = [], 0
        while i < len(rows):
            # keep batch * length bounded, since the scored slice is vocab-wide
            batch, tok_budget = [], 0
            while i < len(rows) and len(batch) < BATCH_ROWS:
                L = len(rows[i])
                if batch and tok_budget + L > BATCH_TOKENS:
                    break
                batch.append(i); tok_budget += L; i += 1
            width = max(len(rows[b]) for b in batch)
            pad = self.tok.pad_token_id
            ids = torch.full((len(batch), width), pad, dtype=torch.long)
            mask = torch.zeros((len(batch), width), dtype=torch.long)
            for k, b in enumerate(batch):
                r = rows[b]
                ids[k, :len(r)] = torch.tensor(r)
                mask[k, :len(r)] = 1
            out += self._score(ids.to(self.dev), mask.to(self.dev),
                               [starts[b] for b in batch])
        return out

    def bits(self, text, condition=""):
        """-log2 P(text | condition). The leading space of `text` belongs to
        `text`, so `condition` must not end in one."""
        if not text:
            return 0.0
        return self.bits_many([text], condition=condition)[0]

    def cost_many(self, programs):
        """c(p) for several programs, batched. Unconditional, so they share no
        context and one forward pass can cover all of them."""
        if not programs:
            return []
        if self._empty is None:
            self._empty = self.bits("\n")
        todo = [p for p in programs if ("p", p) not in self._memo]
        if todo:
            got = self.bits_many([p + "\n" for p in todo])
            for p, v in zip(todo, got):
                self._memo[("p", p)] = v - self._empty
        return [self._memo[("p", p)] for p in programs]

    def cost(self, program):
        """c(p): the program priced on its own, no sequence given.

        This is the frame unquestionability needs. The terminating newline is
        charged so a program pays for stopping, and the empty program is priced
        the same way and subtracted, so c(p_empty) = 0 by construction.

        Unconditional, so it does not depend on the prefix length. That is what
        lets K and every rival be priced ONCE per item instead of once per
        prefix."""
        return self.cost_many([program])[0]

    def cost_given(self, program, shown):
        """c(p | x): the sequence first, the program after. Not used for
        unquestionability. Kept because the quantities in the plan's
        Section 'Quantities measured' include it and S2 needs it."""
        ctx = PRICE_TPL.format(terms=", ".join(shown))
        return (self.bits(" " + program + "\n", condition=ctx)
                - self.bits("\n", condition=ctx))


## 6 · The probes

In [ ]:

# ---- rules, for bits per character and for the exception clause ------------
PROBE_RULES = [
    "the squares, starting at 1",
    "count from 1 in steps of 3",
    "n squared, digits summed",
    "a(1)=1; a(n) = a(n-1) + the digit sum of a(n-1)",
    "n cubed, digits summed, times 4, reversed, last digit",
    "the days of the week in order, repeating",
    "the planets outward from the sun",
    "the NATO alphabet in order",
    "the chemical elements in atomic order, by name",
    "the days of the week, each followed by its position",
    "rule 30 on a ring of 15 cells from one x in the middle, o elsewhere; "
    "each term is the row",
    "an o walks left along ---o; at the end the row grows by one and gains "
    "another o",
    "alternating the eight phases of the moon in order with the NATO alphabet",
    "count n from 1; each term is n in base 4 with digits w, c, s, f",
    "the karate belt colours in order, each reversed, followed by the NATO "
    "word at that position truncated to two letters",
    # added in probe set v2, to widen the search for a cheap clause
    "the primes",
    "count from 1",
    "the letters a to z",
    "the months in order",
    "n times 7",
    "the Fibonacci numbers",
    "the元素 symbols in atomic order".replace("元素", "element "),
    "each term is the previous one reversed",
    "the powers of 2",
    "the counting numbers, each written twice",
]

# ---- the exception clause -------------------------------------------------
# Definition 4.1 caps delta below the cost of the cheapest clause that turns a
# correct program into a rival. A clause costing LESS than delta would sit
# inside budget at every prefix, and nothing could ever certify, so the bound
# has to be a minimum and not an average: a cheaper phrasing we failed to price
# is a real rival we failed to find.
#
# Our minimum is over a finite set, so it can only OVER-estimate the true
# infimum, and that is the unsafe direction. Hence the margin, and hence
# searching harder: probe set v2 has 25 rules and 12 phrasings, 300 probes
# against the 60 of v1, with the terse ones added because the minimum is
# likely to come from them.
CLAUSE_TEMPLATES = [
    "{p}, except term 6 is 0",
    "{p}, but output 0 at term 6",
    "{p}. At term 6 output 0 instead.",
    "{p}, except the 6th term, which is 0",
    # terser phrasings, added in v2
    "{p}; term 6 is 0",
    "{p} but 0 at 6",
    "{p}, with 0 at term 6",
    "{p}, 6th term 0",
    "{p} (term 6: 0)",
    "{p}; 6 -> 0",
    "{p}, except 6",
    "{p} except term 6",
]

_ELEM_SYM = ("h he li be b c n o f ne na mg al si p s cl ar k ca sc ti v cr mn "
             "fe co ni cu zn ga ge as se br kr rb sr y zr").split()
_ELEM_NAME = ("hydrogen helium lithium beryllium boron carbon nitrogen oxygen "
              "fluorine neon sodium magnesium aluminium silicon phosphorus "
              "sulfur chlorine argon potassium calcium scandium titanium "
              "vanadium chromium manganese iron cobalt nickel copper zinc "
              "gallium germanium arsenic selenium bromine krypton rubidium "
              "strontium yttrium zirconium").split()
_COUNTRY = ("albania belgium croatia denmark estonia finland germany hungary "
            "iceland ireland japan kenya latvia mexico norway oman poland qatar "
            "romania sweden turkey uganda vietnam yemen zambia austria bolivia "
            "canada ecuador france ghana india jordan kuwait laos malta nepal "
            "peru serbia tunisia").split()
_STAGES = ("nucleate ram cbd fishing sailing power seaplane restricted "
           "constrained anchored aground towing pushing trawling dredging "
           "surveying mineclearing pilotage").split()

PROBE_SEQUENCES = {
    # the easy end: a machine has the pattern after three terms
    "counting":    [str(n) for n in range(1, 41)],
    "weekdays":    [["monday", "tuesday", "wednesday", "thursday", "friday",
                     "saturday", "sunday"][n % 7] for n in range(40)],
    # middle: predictable but each term still carries something
    "squares":     [str(n * n) for n in range(1, 41)],
    "with-units":  [f"{n} shu" for n in range(1, 41)],
    # the hard end: forty DISTINCT terms out of real tables, where term 30
    # costs what term 5 costs. This is what a battery item actually looks like.
    "elem-symbol": _ELEM_SYM,
    "elem-name":   _ELEM_NAME,
    "country":     _COUNTRY,
    "elem+unit":   [f"{s} pct" for s in _ELEM_SYM],
    "two-table":   [f"{_ELEM_SYM[i]} {_STAGES[i % len(_STAGES)]}"
                    for i in range(40)],
}
PROBE_PREFIXES = [5, 10, 15, 20, 25, 30]   # where each listing is priced

# Which of them the pooled beta is taken from. The generator sizes sequences
# for ITEMS, and an item's terms look like the hard end, not like counting.
# Pooling over all of them would put the value an order of magnitude too low.
BETA_REPRESENTATIVE = ["elem-symbol", "elem-name", "country", "elem+unit",
                       "two-table"]


## 7 · The measurement

In [ ]:
SANITY = [
    ("common English",  "the days of the week in order, repeating"),
    ("scrambled",       "ehT sayd fo eht keew ni redro, gnitaeper"),
    ("random letters",  "qxzjvbhkwmpfgyrtdlncsaeiou qxzjvbhkwmpf"),
]

def sane(ref):
    """Is this machine giving usable prices at all?

    Not every model here is a plain causal LM with a text-only head, and one
    that returns something odd would not fail loudly: it would return numbers,
    they would go into the pooled value, and the generator would size a battery
    on them. So three strings of the same length are priced and the ordering is
    checked. A machine that models text at all charges least for the English
    one and most for the random letters. If it does not, its numbers are not
    prices and the machine is skipped.
    """
    cs = ref.cost_many([s for _, s in SANITY])
    for (lab, _), c in zip(SANITY, cs):
        log(f"      {lab:16s} {c:8.1f} bits")
    if any((c is None) or (c != c) or c <= 0 for c in cs):
        return False, "a price came back non-positive or not a number"
    if not (cs[0] < cs[2]):
        return False, (f"English costs {cs[0]:.0f} and random letters "
                       f"{cs[2]:.0f}; a machine that models text charges less "
                       f"for the English")
    spread = cs[2] - cs[0]
    if spread < 10:
        return False, (f"only {spread:.0f} bits between English and random "
                       f"letters, so it is barely distinguishing them")
    log(f"      ok: {spread:.0f} bits between English and random letters")
    return True, ""

def measure(ref):
    """Every reference constant for one machine, from the fixed probes."""
    out = {}

    # ---- bits per character -----------------------------------------------
    # The generator estimates K by multiplying this by the length of a rule, so
    # what is wanted is the cost of a RULE STRING per character.
    costs = ref.cost_many(PROBE_RULES)
    bpc = sorted(c / len(p) for p, c in zip(PROBE_RULES, costs))
    out["bits_per_char"] = round(st.median(bpc), 3)
    out["bits_per_char_p80"] = round(_q(bpc, 0.80), 3)
    out["bits_per_char_range"] = [round(bpc[0], 2), round(bpc[-1], 2)]
    log(f"    bits per character   median {out['bits_per_char']:.2f}, "
        f"p80 {out['bits_per_char_p80']:.2f}, "
        f"{bpc[0]:.2f} to {bpc[-1]:.2f}")

    # ---- the exception clause, which bounds delta -------------------------
    # The cost of the clause is the INCREMENT over the rule it is bolted onto,
    # so the rule's own cost cancels and what is left is a property of the
    # machine. The minimum over rules and phrasings, because Definition 4.1
    # bounds delta below the CHEAPEST such clause.
    pairs = [(p, tpl.format(p=p)) for p in PROBE_RULES for tpl in CLAUSE_TEMPLATES]
    base = dict(zip(PROBE_RULES, costs))
    full = ref.cost_many([q for _, q in pairs])
    inc = sorted((c - base[p], p) for (p, _), c in zip(pairs, full))
    out["clause_bits"] = round(inc[0][0], 1)
    out["clause_median"] = round(st.median(x for x, _ in inc), 1)
    out["delta_true"] = out["clause_bits"]
    log(f"    exception clause     cheapest {out['clause_bits']:.1f}, "
        f"median {out['clause_median']:.1f}   -> delta_true "
        f"{out['delta_true']:.1f}")

    betas, c0s, per = [], [], {}
    for name, terms in PROBE_SEQUENCES.items():
        strs = [PRINT_TPL + ", ".join(terms[:L]) + LISTING_TAIL
                for L in PROBE_PREFIXES]
        cs = ref.cost_many(strs)
        b, c0, r2 = _fit(PROBE_PREFIXES, cs)
        per[name] = dict(beta=round(b, 2), c0=round(c0, 1), r2=round(r2, 3))
        if r2 >= BETA_MIN_R2 and name in BETA_REPRESENTATIVE:
            betas.append(b); c0s.append(c0)
        _rep = "  representative" if name in BETA_REPRESENTATIVE else ""
        log(f"      {name:12s} beta {b:6.2f}  c0 {c0:7.1f}  R2 {r2:.2f}"
            + ("" if r2 >= BETA_MIN_R2 else "   (R2 too low)") + _rep)
    out["beta_by_sequence"] = per
    if betas:
        out["beta_listing"] = round(st.median(betas), 2)
        out["beta_low"] = round(min(betas), 2)
        out["beta_high"] = round(max(betas), 2)
        out["c0_listing"] = round(st.median(c0s), 1)
        log(f"    beta                 median {out['beta_listing']:.2f}, "
            f"{out['beta_low']:.2f} to {out['beta_high']:.2f} across "
            f"{len(betas)} of the representative sequences")
        log(f"                         pooled over the ones whose terms are "
            f"forty distinct things, because that is what an item looks like."
            )
        log(f"                         Counting and weekdays are priced too, "
            f"and reported, but a value taken from them would be an order of")
        log(f"                         magnitude low.")
        log(f"    c0                   median {out['c0_listing']:.1f}")
    else:
        log(f"    beta                 no sequence fitted above R2 "
            f"{BETA_MIN_R2}")

    # ---- the empty wrapper, for comparison --------------------------------
    # c0 above is a fitted intercept. This is the wrapper priced directly, with
    # no terms in it. They are not the same thing and the gap is worth seeing:
    # the fitted one absorbs whatever the first terms cost.
    out["wrapper_bits"] = round(ref.cost_many(
        [PRINT_TPL.rstrip() + LISTING_TAIL])[0], 1)
    log(f"    the wrapper alone    {out['wrapper_bits']:.1f} bits   "
        f"(c0 fitted is {out.get('c0_listing', float('nan')):.1f})")
    out["n_probes"] = len(PROBE_RULES) + len(pairs) + \
        len(PROBE_SEQUENCES) * len(PROBE_PREFIXES)
    return out


## 8 · Run

In [ ]:
stage("MEASURING")
results = {}
for n, name in enumerate(MACHINES, 1):
    log(f"\n{'='*72}\n[{n}/{len(MACHINES)}] {name}\n{'='*72}")
    _free = _disk_free_gb()
    log(f"  disk free {_free:.0f} GB")
    if _free < DISK_MIN_GB:
        log(f"  under DISK_MIN_GB = {DISK_MIN_GB}, skipping. Clear the cache "
            f"or drop a machine from MACHINES.")
        continue
    try:
        t0 = time.time()
        ref = Machine(name, four_bit=LOAD_4BIT)
        log(f"    checking it gives usable prices")
        _ok, _why = sane(ref)
        if not _ok:
            log(f"    SKIPPED: {_why}")
            log(f"    Its numbers would have gone into the pooled value and "
                f"sized a battery, so it is better to lose the machine.")
            raise RuntimeError("failed the sanity check")
        results[name] = measure(ref)
        results[name].update(machine=name, four_bit=LOAD_4BIT,
                             probes=PROBE_SET_ID, measured_by=BUILD,
                             measured_on=datetime.datetime.now()
                             .isoformat(timespec="minutes"))
        log(f"  {results[name]['n_probes']} probes in {time.time()-t0:.0f}s")
        # Written NOW, not at the end. Four machines is forty minutes with a
        # download each, and a session that dies on the third would otherwise
        # lose the first two.
        _save_reference({name: results[name]})
        log(f"  saved")
    except Exception as e:
        log(f"  FAILED: {str(e)[:160]}")
        log(f"  The others carry on. What is already measured is on disk.")
    finally:
        # One machine at a time. A T4 session has about 21 GB of disk and an
        # 8B model in 4-bit needs most of it, so the weights have to go before
        # the next download starts.
        for v in ("ref",):
            if v in dir(): pass
        try:
            del ref
        except Exception:
            pass
        _unload()
        log(f"  unloaded, disk free {_disk_free_gb():.0f} GB")

# ---------------------------------------------------------------------------
stage("POOLING")
if not results:
    log("  nothing measured.")
else:
    log(f"  {len(results)} machine(s): " + ", ".join(results))
    log(f"\n  {'':22s}" + "".join(f"{k.split('/')[-1][:14]:>16s}"
                                  for k in results))
    for f, fmt in (("bits_per_char", "{:.2f}"), ("bits_per_char_p80", "{:.2f}"),
                   ("clause_bits", "{:.1f}"), ("beta_listing", "{:.2f}"),
                   ("beta_low", "{:.2f}"), ("c0_listing", "{:.1f}"),
                   ("wrapper_bits", "{:.1f}")):
        row = "".join(f"{fmt.format(results[k][f]):>16s}" if f in results[k]
                      else f"{'-':>16s}" for k in results)
        log(f"  {f:22s}{row}")

    # The pooled value takes the end of each that asks for MORE TERMS, not the
    # mean. The errors are not symmetric: a sequence longer than it needed to
    # be costs a little generation time, while one that is too short cannot be
    # repaired without regenerating the item.
    #
    # Note this is about SIZING, not about the tolerance. delta_gen is pooled
    # high here because a larger delta in the sizing formula asks for a longer
    # prefix. The tolerance a pricing run uses is a separate choice, capped by
    # Definition 4.1 and free below it, and a LOWER one there is more
    # permissive: fewer rivals fall inside the budget, so more items certify.
    def _pick(field, how):
        vals = [r[field] for r in results.values() if field in r]
        return round(how(vals), 2) if vals else None

    pooled = dict(
        bits_per_char=_pick("bits_per_char_p80", max),
        beta_listing=_pick("beta_low", min),
        c0_listing=_pick("c0_listing", min),
        delta_gen=_pick("clause_bits", max),
        machines=list(results),
        probes=PROBE_SET_ID,
        measured_on=datetime.datetime.now().isoformat(timespec="minutes"))
    log(f"\n  POOLED, for the generator:")
    for k in ("bits_per_char", "beta_listing", "c0_listing", "delta_gen"):
        log(f"    {k:16s} {pooled[k]}")
    log(f"\n  These size sequences and choose rival prefixes. They buy "
        f"EFFICIENCY, not validity: with an unlimited budget the generator")
    log(f"  would ask for the longest sequence the executor holds and for "
        f"rivals at every prefix, and consult none of them. Nothing in")
    log(f"  Definition 4.1 or Proposition 4.1 refers to them.")
    log(f"\n  delta_gen is NOT the delta unquestionability uses. That one is "
        f"bounded by the cheapest exception clause UNDER THE MACHINE DOING")
    log(f"  THE PRICING, so it is per machine and sits in each machine's own "
        f"block as delta_true.")

    # ---- spread, which is the point of using several machines -------------
    for f in ("clause_bits", "beta_listing", "bits_per_char"):
        v = [r[f] for r in results.values() if f in r]
        if len(v) > 1:
            log(f"\n  {f}: {min(v):.2f} to {max(v):.2f} across "
                f"{len(v)} machines, spread {max(v)/max(min(v),1e-9)-1:+.0%}")
            log(f"    A narrow spread means the constant belongs to the "
                f"reference machines as a class and a battery sized by it is "
                f"not tilted")
            log(f"    towards any one of them. A wide one means it does not, "
                f"and the plan should say so.")


## 9 · Write

In [ ]:
stage("WRITING ctest_calibration.json")

# Each machine was already saved as it finished. This settles the pooled block.
if not results:
    log("  nothing measured, so nothing to pool.")
else:
    _ALL = _load_calibration()
    _refs = {k: v["reference"] for k, v in _ALL.get("machines", {}).items()
             if "reference" in v}
    _refs.update(results)

    if POOL_MACHINES is not None:
        _named = {k: r for k, r in _refs.items() if k in POOL_MACHINES}
        _out = sorted(set(_refs) - set(_named))
        if _out:
            log(f"  pooling over the {len(_named)} named machine(s); "
                f"{len(_out)} calibrated but not pooled: {', '.join(_out)}")
        _missing = sorted(set(POOL_MACHINES) - set(_refs))
        if _missing:
            log(f"  WARNING: {len(_missing)} named machine(s) have no block on "
                f"this probe set and are missing from the pool: "
                f"{', '.join(_missing)}")
        _refs = _named

    # Never pool across probe sets. A block from an older set is not the same
    # measurement: set v1's sequences were all cycles or counts and its beta
    # came out an order of magnitude low, so a value mixing the two would be
    # neither one thing nor the other.
    _stale = {k: r.get("probes") for k, r in _refs.items()
              if r.get("probes") != PROBE_SET_ID}
    if _stale:
        log(f"  {len(_stale)} machine(s) carry an older probe set and are left "
            f"out of the pool:")
        for k, v in _stale.items():
            log(f"      {k}   probes {v!r}, this run is {PROBE_SET_ID!r}")
        log(f"    Re-run them with CALIB_REMEASURE = True to bring them in.")
        _refs = {k: r for k, r in _refs.items()
                 if r.get("probes") == PROBE_SET_ID}

    if not _refs:
        log(f"  nothing on {PROBE_SET_ID} to pool from.")
    else:
        def _pick_all(field, how):
            vals = [r[field] for r in _refs.values() if field in r]
            return round(how(vals), 2) if vals else None

        pooled = dict(
            bits_per_char=_pick_all("bits_per_char_p80", max),
            # the MEDIAN of each machine's representative sequences, then the
            # smallest of those. Not the minimum of minima: that took the one
            # worst-fitting sequence on the one machine where it fitted worst,
            # 0.98 from elem-name on Llama, and a generator sized on it would
            # have asked for a 140-term prefix for a 70-character rule. The
            # median is unmoved by one bad probe among five, and lands at 4.88
            # against the 3.40 the pricing pass measures on real items.
            beta_listing=_pick_all("beta_listing", min),
            beta_gate=_pick_all("beta_high", max),
            c0_listing=_pick_all("c0_listing", min),
            delta_gen=_pick_all("clause_bits", max),
            machines=sorted(_refs), probes=PROBE_SET_ID,
            measured_on=datetime.datetime.now().isoformat(timespec="minutes"))
        log(f"  pooled over {len(_refs)} machine(s) on {PROBE_SET_ID}:")
        for k in ("bits_per_char", "beta_listing", "c0_listing", "delta_gen"):
            log(f"    {k:16s} {pooled[k]}")

    ALL = _save_reference(results, pooled=pooled if _refs else None)
    log(f"\n  wrote {CALIBRATION}")
    log(f"    machines with a reference block: "
        f"{sum(1 for v in ALL['machines'].values() if 'reference' in v)}")
    log(f"    machines with fitted blocks:     "
        f"{sum(1 for v in ALL['machines'].values() if 'fitted' in v)}")
    log(f"    pooled from: {', '.join(pooled['machines'])}")
    log(f"\n  The generator reads `pooled` and falls back to its config "
        f"defaults when the file is absent, so a run without this notebook")
    log(f"  still works; it simply uses numbers somebody measured earlier and "
        f"hardcoded.")

state_save(f"{BUILD}: reference constants for {len(results)} machine(s)")
